<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/main/notebooks/03_phase2/SWIN_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Image Model: SWIN V2

In [1]:
!pip install -q timm transformers accelerate

In [2]:
!pip install -q timm transformers accelerate wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00


In [3]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms
import matplotlib.pyplot as plt

import wandb
import open_clip
from huggingface_hub import hf_hub_download

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [6]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
LOCAL_CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
LOCAL_IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

CSV_PATH = LOCAL_CSV_PATH
IMAGE_DIR = LOCAL_IMAGE_DIR

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


In [7]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 256,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "swinv2_tiny_window8_256",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 1.0
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

Data Loading and Text Cleaning

In [9]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_1688/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_1688/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_1688/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [10]:
df["hybrid_gemma3_4b_clean"].head(10)

,hybrid_gemma3_4b_clean
0,the image depicts a landscape dominated by ext...
1,the image depicts a largely arid landscape dom...
2,the image depicts a landscape dominated by ext...
3,the image depicts a valley dominated by dense ...
4,the image depicts a coastal area dominated by ...
5,the image depicts a landscape dominated by cul...
6,the image depicts a landscape dominated by a r...
7,the image depicts a hilly landscape dominated ...
8,the image depicts a landscape dominated by ext...
9,the image depicts a landscape dominated by ext...


In [11]:
df["hybrid_gemma3_4b_clean"].str.contains(
    r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
    regex=True,
    case=False
).sum()

/tmp/ipykernel_1688/1445695331.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["hybrid_gemma3_4b_clean"].str.contains(


np.int64(0)

Stratified Sampling and Split

In [12]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

sample_df = sample_df.reset_index(drop=True)

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


In [13]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [14]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


Transforms

In [15]:
IMG_SIZE = CONFIG["img_size"]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Dataset Classes

In [16]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=128):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [17]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

Swin image-only

In [18]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=7):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

Swin + MiniLM

In [19]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=7,
        text_scale=1.0
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

Remote CLIP Text Encoder

In [20]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename="RemoteCLIP-ViT-B-32.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [21]:
class SwinV2RemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=7,
        text_scale=1.0

    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

Evaluation functions

In [22]:
@torch.no_grad()
def evaluate_image_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

In [23]:
@torch.no_grad()
def evaluate_text_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        input_ids = batch["input_ids"].to(config["device"])
        attention_mask = batch["attention_mask"].to(config["device"])
        targets = batch["target"].to(config["device"])

        preds = model(images, input_ids, attention_mask)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

In [24]:
@torch.no_grad()
def evaluate_remoteclip_model(model, loader, config):
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images = batch["image"].to(config["device"])
        texts = batch["text"]
        targets = batch["target"].to(config["device"])

        preds = model(images, texts)
        loss = criterion(preds, targets)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae

W&B Training Functions

In [25]:
def train_image_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_image_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [26]:
def train_text_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            input_ids = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            targets = batch["target"].to(config["device"])

            preds = model(images, input_ids, attention_mask)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_text_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [27]:
def train_remoteclip_model(model, train_loader, val_loader, config):
    criterion = nn.L1Loss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images = batch["image"].to(config["device"])
            texts = batch["text"]
            targets = batch["target"].to(config["device"])

            preds = model(images, texts)
            loss = criterion(preds, targets)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _ = evaluate_remoteclip_model(model, val_loader, config)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [28]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Image Only Model

In [29]:
def run_swin_image_only():
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "swinv2_image_only"
    run_config["fusion"] = "none"
    run_config["text_column"] = "none"
    run_config["text_model"] = "none"

    wandb.init(
        project="DI725-Project-landcover-regression_phase2",
        name="swinv2_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history = train_image_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae = evaluate_image_model(model, test_loader, CONFIG)

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
        "experiment": "swinv2_image_only",
        "test_mae": test_mae,
        "test_rmse": test_rmse
    }

In [30]:
def run_swin_transformer_text(text_col, text_model, text_scale=1.0):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "swinv2_transformer_text"
    run_config["text_column"] = text_col
    run_config["text_model"] = text_model
    run_config["text_scale"] = text_scale
    run_config["fusion"] = "gated_frozen_scaled"

    run_name = f"swinv2_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(
        project="DI725-Project-landcover-regression_phase2",
        name=run_name,
        config=run_config,
        reinit=True
    )

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds = LandCoverDataset(val_df, IMAGE_DIR, transform=eval_tfms, tokenizer=tokenizer, text_col=text_col)
    test_ds = LandCoverDataset(test_df, IMAGE_DIR, transform=eval_tfms, tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history = train_text_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae = evaluate_text_model(model, test_loader, CONFIG)

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
        "experiment": run_name,
        "text_column": text_col,
        "text_model": text_model,
        "text_scale": text_scale,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    }

In [31]:
def run_swin_remoteclip_text(text_col, text_scale=1.0):
    seed_everything(CONFIG["seed"])

    run_config = CONFIG.copy()
    run_config["experiment"] = "swinv2_remoteclip_text"
    run_config["text_column"] = text_col
    run_config["text_model"] = "RemoteCLIP ViT-B-32"
    run_config["text_scale"] = text_scale
    run_config["fusion"] = "gated_frozen_scaled"

    run_name = f"swinv2_remoteclip_{text_col}_scale_{text_scale}"

    wandb.init(
        project="DI725-Project-landcover-regression_phase2",
        name=run_name,
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverRawTextDataset(train_df, IMAGE_DIR, transform=train_tfms, text_col=text_col)
    val_ds = LandCoverRawTextDataset(val_df, IMAGE_DIR, transform=eval_tfms, text_col=text_col)
    test_ds = LandCoverRawTextDataset(test_df, IMAGE_DIR, transform=eval_tfms, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = SwinV2RemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history = train_remoteclip_model(model, train_loader, val_loader, CONFIG)

    test_loss, test_mae, test_rmse, class_mae = evaluate_remoteclip_model(model, test_loader, CONFIG)

    wandb.log({
        "test_loss": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({
                "class": TARGET_COLS,
                "class_mae": class_mae
            })
        )
    })

    wandb.finish()

    return {
        "experiment": run_name,
        "text_column": text_col,
        "text_model": "RemoteCLIP ViT-B-32",
        "text_scale": text_scale,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    }

Run Experiments

In [32]:
results = []

In [33]:
results.append(run_swin_image_only())
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 10.9682 | Val MAE: 7.7778 | Val RMSE: 17.7582
Epoch 2/15 | Train Loss: 6.7430 | Val MAE: 4.9107 | Val RMSE: 11.5146
Epoch 3/15 | Train Loss: 5.0898 | Val MAE: 3.5113 | Val RMSE: 8.9017
Epoch 4/15 | Train Loss: 4.3713 | Val MAE: 3.0955 | Val RMSE: 7.8415
Epoch 5/15 | Train Loss: 4.0087 | Val MAE: 2.9009 | Val RMSE: 7.6844
Epoch 6/15 | Train Loss: 3.6747 | Val MAE: 2.7835 | Val RMSE: 7.5142
Epoch 7/15 | Train Loss: 3.4191 | Val MAE: 2.6937 | Val RMSE: 7.2821
Epoch 8/15 | Train Loss: 3.3046 | Val MAE: 2.5172 | Val RMSE: 6.7755
Epoch 9/15 | Train Loss: 3.1178 | Val MAE: 2.5765 | Val RMSE: 6.9001
Epoch 10/15 | Train Loss: 2.9318 | Val MAE: 2.4588 | Val RMSE: 6.2626
Epoch 11/15 | Train Loss: 2.8181 | Val MAE: 2.4280 | Val RMSE: 6.5568
Epoch 12/15 | Train Loss: 2.6824 | Val MAE: 2.3277 | Val RMSE: 6.2206
Epoch 13/15 | Train Loss: 2.5373 | Val MAE: 2.2420 | Val RMSE: 5.9037
Epoch 14/15 | Train Loss: 2.5281 | Val MAE: 2.3798 | Val RMSE: 6.5280
Epoch 15/15 | Train Loss: 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
epoch,15
test_loss,2.4598
test_mae,2.4598


,experiment,test_mae,test_rmse
0,swinv2_image_only,2.459801,6.409931


MiniLM Experiments

In [34]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

In [36]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_swin_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=1.0
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 10.7399 | Val MAE: 7.5796 | Val RMSE: 17.0976
Epoch 2/15 | Train Loss: 6.5551 | Val MAE: 4.3771 | Val RMSE: 10.4659
Epoch 3/15 | Train Loss: 4.9152 | Val MAE: 3.6891 | Val RMSE: 9.4631
Epoch 4/15 | Train Loss: 4.2882 | Val MAE: 3.0661 | Val RMSE: 7.9689
Epoch 5/15 | Train Loss: 3.9551 | Val MAE: 3.0085 | Val RMSE: 8.0658
Epoch 6/15 | Train Loss: 3.6774 | Val MAE: 2.7501 | Val RMSE: 7.2918
Epoch 7/15 | Train Loss: 3.4922 | Val MAE: 3.0314 | Val RMSE: 8.0438
Epoch 8/15 | Train Loss: 3.2973 | Val MAE: 2.6513 | Val RMSE: 6.9028
Epoch 9/15 | Train Loss: 3.0910 | Val MAE: 2.4876 | Val RMSE: 6.4677
Epoch 10/15 | Train Loss: 2.9895 | Val MAE: 2.6291 | Val RMSE: 7.1190
Epoch 11/15 | Train Loss: 2.8630 | Val MAE: 2.6608 | Val RMSE: 6.2756
Epoch 12/15 | Train Loss: 2.6529 | Val MAE: 2.5350 | Val RMSE: 6.2552
Early stopping triggered.


epoch,▁▂▂▃▄▄▅▅▆▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▂▂▂▂▂▁▁▁▁
val_loss,█▄▃▂▂▁▂▁▁▁▁▁
val_mae,█▄▃▂▂▁▂▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▁▁▂▁▁
epoch,12
test_loss,2.71291
test_mae,2.71291


,experiment,test_mae,test_rmse,text_column,text_model,text_scale
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,1.834890,4.304175,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.223753,5.417362,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0
0,swinv2_image_only,2.459801,6.409931,NaN,NaN,NaN
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.712907,6.667156,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0


DistilBERT Experiments

In [37]:
CONFIG["text_model"] = "distilbert-base-uncased"

In [38]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_swin_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=1.0
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.0105 | Val MAE: 7.7835 | Val RMSE: 17.6970
Epoch 2/15 | Train Loss: 6.5413 | Val MAE: 4.2718 | Val RMSE: 10.4768
Epoch 3/15 | Train Loss: 4.8413 | Val MAE: 3.4711 | Val RMSE: 8.6680
Epoch 4/15 | Train Loss: 4.2557 | Val MAE: 3.0355 | Val RMSE: 8.0319
Epoch 5/15 | Train Loss: 3.8611 | Val MAE: 3.0079 | Val RMSE: 8.1649
Epoch 6/15 | Train Loss: 3.6827 | Val MAE: 2.9245 | Val RMSE: 7.8646
Epoch 7/15 | Train Loss: 3.4082 | Val MAE: 2.8249 | Val RMSE: 7.5513
Epoch 8/15 | Train Loss: 3.2257 | Val MAE: 2.6242 | Val RMSE: 7.3565
Epoch 9/15 | Train Loss: 3.0248 | Val MAE: 2.4359 | Val RMSE: 6.5356
Epoch 10/15 | Train Loss: 2.8590 | Val MAE: 2.6439 | Val RMSE: 7.1664
Epoch 11/15 | Train Loss: 2.7340 | Val MAE: 2.6436 | Val RMSE: 7.1214
Epoch 12/15 | Train Loss: 2.6543 | Val MAE: 2.2515 | Val RMSE: 6.0096
Epoch 13/15 | Train Loss: 2.4833 | Val MAE: 2.3022 | Val RMSE: 6.1255
Epoch 14/15 | Train Loss: 2.3903 | Val MAE: 2.3095 | Val RMSE: 6.1731
Epoch 15/15 | Train Loss: 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▄▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▂▂▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▂▂▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▁▂▂▁▁▁▁
epoch,15
test_loss,2.32795
test_mae,2.32795


,experiment,test_mae,test_rmse,text_column,text_model,text_scale
5,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,1.818062,4.304616,text_qwen3_4b_clean,distilbert-base-uncased,1.0
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,1.834890,4.304175,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.223753,5.417362,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0
4,swinv2_hybrid_gemma3_4b_clean_distilbert-base-...,2.271163,5.696041,hybrid_gemma3_4b_clean,distilbert-base-uncased,1.0
6,swinv2_vision_gemma3_4b_clean_distilbert-base-...,2.327954,6.173319,vision_gemma3_4b_clean,distilbert-base-uncased,1.0
0,swinv2_image_only,2.459801,6.409931,NaN,NaN,NaN
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.712907,6.667156,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0


Experiment Summary

In [40]:
results_df = pd.DataFrame(results).sort_values("test_mae")
results_df

,experiment,test_mae,test_rmse,text_column,text_model,text_scale
5,swinv2_text_qwen3_4b_clean_distilbert-base-unc...,1.818062,4.304616,text_qwen3_4b_clean,distilbert-base-uncased,1.0
2,swinv2_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,1.834890,4.304175,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0
1,swinv2_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,2.223753,5.417362,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0
4,swinv2_hybrid_gemma3_4b_clean_distilbert-base-...,2.271163,5.696041,hybrid_gemma3_4b_clean,distilbert-base-uncased,1.0
6,swinv2_vision_gemma3_4b_clean_distilbert-base-...,2.327954,6.173319,vision_gemma3_4b_clean,distilbert-base-uncased,1.0
0,swinv2_image_only,2.459801,6.409931,NaN,NaN,NaN
3,swinv2_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,2.712907,6.667156,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,1.0
